In [3]:
import torch

# following OSS configs
B, S, H = 1, 10, 2880
intermediate_size = H
top_k_experts = 4
num_experts = 128


In [13]:
class MoEBlock(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = torch.nn.Linear(H, num_experts)

        self.mlp1_weight = torch.nn.Parameter(torch.randn(num_experts, H, 2*intermediate_size))
        self.mlp1_bias = torch.nn.Parameter(torch.randn(num_experts, 2*intermediate_size))

        self.mlp2_weight = torch.nn.Parameter(torch.randn(num_experts, intermediate_size, H))
        self.mlp2_bias = torch.nn.Parameter(torch.randn(num_experts, H))

    def forward(self, x):
        # omitted pre-norm
        out = self.gate(x) 
        expert_weights, expert_indices = torch.topk(out, top_k_experts)
        expert_weights = torch.nn.functional.softmax(expert_weights, dim=-1)
        print(expert_weights.shape)

        expert_mlp1_weight, expert_mlp1_bias = self.mlp1_weight[expert_indices, ...], self.mlp1_bias[expert_indices, ...]

        t = torch.einsum('sh,sehi->sei', x, expert_mlp1_weight) + expert_mlp1_bias

        gate, up = t[..., ::2], t[..., 1::2]
        t = torch.nn.functional.silu(gate) * up

        expert_mlp2_weight, expert_mlp2_bias = self.mlp2_weight[expert_indices, ...], self.mlp2_bias[expert_indices, ...]

        t = torch.einsum('sei,seih->seh', t, expert_mlp2_weight) + expert_mlp2_bias
        
        t = torch.einsum('se,seh->sh', expert_weights, t)
        return t


moe = MoEBlock()
x = torch.randn(S, H) # assume flattened batch index
moe(x)

torch.Size([10, 4])


tensor([[  87725.9219,   -6089.5234,   75225.3047,  ...,  -31857.5703,
          -25626.1953,  -73791.1484],
        [ -20191.2344,  107579.8984,  -82916.4219,  ...,    2357.4854,
          -78726.8438,   23573.2754],
        [  50603.3828,  -47992.6484,   -3964.8662,  ...,   95346.0625,
           62240.3359, -134156.7031],
        ...,
        [ -34537.7734,   22623.8535, -119822.9219,  ...,  -37706.2500,
           -9685.6094,   23288.9570],
        [ -45739.0000,  -66545.7578,     892.2354,  ...,    4429.8164,
           -5764.1680,  -33465.1953],
        [ -46498.4453,   69269.3359,   -7464.0625,  ...,   21670.9375,
          -36210.5078,  -37133.8633]], grad_fn=<ViewBackward0>)